# 01 - Exploratory Data Analysis: Orders

This notebook explores the orders dataset to understand delivery patterns and identify potential fraud indicators.

## Objectives
- Understand the structure and quality of orders data
- Analyze order distribution by region, time, and value
- Identify patterns in missing items
- Detect anomalies and outliers

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sys
sys.path.insert(0, '..')

from src.etl.extractors import extract_orders
from src.etl.transformers import transform_orders
from src.features.order_features import create_order_features
from src.analysis.descriptive import describe_orders

# Settings
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load and Transform Data

In [ ]:
# Load raw data
orders_raw = extract_orders()
print(f"Raw orders shape: {orders_raw.shape}")
orders_raw.head()

In [ ]:
# Transform data
orders = transform_orders(orders_raw)
print(f"Transformed orders shape: {orders.shape}")
orders.head()

In [ ]:
# Check data types
orders.dtypes

## 2. Basic Statistics

In [ ]:
# Descriptive statistics
stats = describe_orders(orders)
for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
# Numeric columns summary
orders.describe()

In [ ]:
# Missing values
missing = orders.isnull().sum()
missing[missing > 0]

## 3. Order Amount Analysis

In [ ]:
# Order amount distribution
fig = px.histogram(
    orders, 
    x='order_amount', 
    nbins=50,
    title='Distribution of Order Amounts',
    labels={'order_amount': 'Order Amount ($)'}
)
fig.show()

In [ ]:
# Order amount by region
fig = px.box(
    orders, 
    x='region', 
    y='order_amount',
    title='Order Amount by Region',
    labels={'order_amount': 'Order Amount ($)', 'region': 'Region'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 4. Missing Items Analysis

In [ ]:
# Add features
orders_feat = create_order_features(orders)

In [ ]:
# Missing items distribution
fig = px.histogram(
    orders_feat, 
    x='items_missing',
    title='Distribution of Missing Items per Order',
    labels={'items_missing': 'Number of Missing Items'}
)
fig.show()

In [ ]:
# Missing rate distribution
fig = px.histogram(
    orders_feat[orders_feat['has_missing']], 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate (Orders with Missing Items Only)',
    labels={'missing_rate': 'Missing Rate (%)'}
)
fig.show()

In [ ]:
# Orders with vs without missing items
missing_counts = orders_feat['has_missing'].value_counts()
fig = px.pie(
    values=missing_counts.values, 
    names=['No Missing Items', 'Has Missing Items'],
    title='Orders with Missing Items'
)
fig.show()

## 5. Regional Analysis

In [ ]:
# Orders by region
regional = orders_feat.groupby('region').agg({
    'order_id': 'count',
    'items_missing': 'sum',
    'items_delivered': 'sum',
    'order_amount': 'sum'
}).reset_index()
regional.columns = ['region', 'total_orders', 'items_missing', 'items_delivered', 'total_revenue']
regional['missing_rate'] = regional['items_missing'] / (regional['items_delivered'] + regional['items_missing']) * 100
regional.sort_values('missing_rate', ascending=False)

In [ ]:
# Missing rate by region
fig = px.bar(
    regional.sort_values('missing_rate', ascending=True), 
    x='missing_rate', 
    y='region',
    orientation='h',
    title='Missing Rate by Region',
    labels={'missing_rate': 'Missing Rate (%)', 'region': 'Region'},
    color='missing_rate',
    color_continuous_scale='Reds'
)
fig.show()

## 6. Temporal Analysis

In [ ]:
# Monthly trends
orders_feat['month'] = pd.to_datetime(orders_feat['order_date']).dt.to_period('M')
monthly = orders_feat.groupby('month').agg({
    'order_id': 'count',
    'items_missing': 'sum',
    'items_delivered': 'sum'
}).reset_index()
monthly['missing_rate'] = monthly['items_missing'] / (monthly['items_delivered'] + monthly['items_missing']) * 100
monthly['month'] = monthly['month'].astype(str)

In [ ]:
# Plot monthly trends
fig = make_subplots(rows=2, cols=1, subplot_titles=('Total Orders', 'Missing Rate'))

fig.add_trace(
    go.Bar(x=monthly['month'], y=monthly['order_id'], name='Orders'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=monthly['month'], y=monthly['missing_rate'], mode='lines+markers', name='Missing Rate'),
    row=2, col=1
)

fig.update_layout(height=600, title_text='Monthly Trends')
fig.show()

In [ ]:
# Day of week analysis
orders_feat['day_of_week'] = pd.to_datetime(orders_feat['order_date']).dt.day_name()
daily = orders_feat.groupby('day_of_week').agg({
    'order_id': 'count',
    'items_missing': 'sum',
    'items_delivered': 'sum'
}).reset_index()
daily['missing_rate'] = daily['items_missing'] / (daily['items_delivered'] + daily['items_missing']) * 100

# Sort by day
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily['day_of_week'] = pd.Categorical(daily['day_of_week'], categories=day_order, ordered=True)
daily = daily.sort_values('day_of_week')

fig = px.bar(
    daily, 
    x='day_of_week', 
    y='missing_rate',
    title='Missing Rate by Day of Week',
    labels={'missing_rate': 'Missing Rate (%)', 'day_of_week': 'Day of Week'},
    color='missing_rate',
    color_continuous_scale='Reds'
)
fig.show()

## 7. Correlation Analysis

In [ ]:
# Correlation between numeric features
numeric_cols = ['order_amount', 'items_delivered', 'items_missing', 'total_items', 'missing_rate']
corr = orders_feat[numeric_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    title='Correlation Matrix',
    color_continuous_scale='RdBu_r'
)
fig.show()

## 8. Key Findings

### Summary
- **Total Orders**: [Fill after running]
- **Overall Missing Rate**: [Fill after running]
- **Highest Risk Region**: [Fill after running]
- **Peak Fraud Period**: [Fill after running]

### Insights
1. [Add insights after analysis]
2. [Add insights after analysis]
3. [Add insights after analysis]